# NemoIR SLM Autoresearch — MNLI LoRA Viewer Notebook

**Pre-rendered on GitHub:** the full trace below was captured from a live 25-trial run (base 0.3415 → best 0.7405). No install, GPU, or API key needed to *view*.

- **Workflow:** `autoresearch.nemo` — hill-climbing over `candidate.py` (LoRA rank/alpha/dropout, LR, steps, prompt) with frozen `harness/*` evaluator; compiled numeric guard `score - best > eps`.
- **Local run:** `git clone https://github.com/hkalexling/nemoir && cd public/demos/slm-autoresearch && python -m venv .venv && .venv/bin/pip install -e . && .venv/bin/pip install bitsandbytes && cp .env.example .env && $EDITOR .env` then `.venv/bin/python run.py --model $NEMOIR_MODEL --max-trials 5  # any LiteLLM model, e.g. openai/gpt-4o-mini, deepseek/deepseek-chat`
- **Colab path:** cells below use `gdown` to fetch a Drive zip — **skip on local checkout** (you already have the code).
- **Viewer note:** outputs are intentionally kept so the notebook is useful as documentation; the last cell's log is the audit trail (`runs/current/trial_history.jsonl`).

> Badge: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hkalexling/nemoir/blob/main/public/demos/slm-autoresearch/demo.ipynb)


In [ ]:
# Colab bootstrap — skip on local checkout (you already have public/demos/slm-autoresearch/)
!gdown --id 12jkHcldlvjyegJChLoJnjr-rSrHYSbuB

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=12jkHcldlvjyegJChLoJnjr-rSrHYSbuB
To: /content/slm-autoresearch.zip
100% 40.9k/40.9k [00:00<00:00, 66.8MB/s]


In [ ]:
# Colab: unzip the fetched archive — skip locally
!unzip slm-autoresearch.zip

Archive:  slm-autoresearch.zip
  inflating: autoresearch/_agent.py  
 extracting: autoresearch.egg-info/dependency_links.txt  
  inflating: autoresearch.egg-info/PKG-INFO  
 extracting: autoresearch.egg-info/requires.txt  
  inflating: autoresearch.egg-info/SOURCES.txt  
 extracting: autoresearch.egg-info/top_level.txt  
  inflating: autoresearch.html       
  inflating: autoresearch/__init__.py  
  inflating: autoresearch/_manifest.py  
  inflating: autoresearch.nemo       
  inflating: autoresearch/types.py   
  inflating: candidate.py            
 extracting: .env.example            
  inflating: .gitignore              
  inflating: harness/benchmarks.yml  
  inflating: harness/eval.py         
 extracting: harness/__init__.py     
  inflating: harness/judge.py        
  inflating: harness/preflight.py    
  inflating: harness/state.py        
  inflating: harness_tools.py        
  inflating: harness/train.py        
  inflating: pyproject.toml          
  inflating: run.py       

In [ ]:
# Colab deps — locally: pip install -e . && pip install bitsandbytes (see README)
!pip install -e . && pip install bitsandbytes

Obtaining file:///content
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 29.2 MB/s eta 0:00:00
  Building editable for autoresearch (pyproject.toml) ... done
  Created wheel for autoresearch: filename=autoresearch-0.1.0-0.editable-py3-none-any.whl size=2728 sha256=f79fd46932aae20bf0e7eeb941b3ceef6aad03082c7f3998c18eb7fb81780c05
  Stored in directory: /tmp/pip-ephem-wheel-cache-8fccspdw/wheels/bd/e2/ad/6557ae2989fbf3d2351bffa42147f9477243538a6ea9803db9
Successfully built autoresearch
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib_metadata 9.0.0
    Uninstalling importlib_meta

In [ ]:
# Run — Colab uses userdata.get(); locally, .env / env var works too (see README)
# Any LiteLLM model/provider works (like xgboost/cvxpygen): set NEMOIR_MODEL + NEMOIR_API_KEY
from google.colab import userdata
import os
os.environ['NEMOIR_API_KEY'] = os.environ.get('NEMOIR_API_KEY') or os.environ.get('OPENAI_API_KEY') or os.environ.get('DEEPSEEK_API_KEY') or userdata.get('NEMOIR_API_KEY') or userdata.get('OPENAI_API_KEY') or userdata.get('DEEPSEEK_API_KEY') or ""
os.environ['NEMOIR_MODEL'] = os.environ.get('NEMOIR_MODEL') or "openai/gpt-4o-mini"
# Legacy DEEPSEEK_API_KEY also works via the fallback above
print('NEMOIR_MODEL:', os.environ.get('NEMOIR_MODEL'))
print('NEMOIR_API_KEY env set:', bool(os.environ.get('NEMOIR_API_KEY')))

!python run.py --model $NEMOIR_MODEL --max-trials 25


DEEPSEEK_API_KEY env set: True
Autoresearch MNLI: model=deepseek-v4-pro cwd=/content eps=0.01 profile=mnli_demo max_trials=25 train_examples=candidate.py eval_examples=benchmarks.yml train_timeout_seconds=1140

-- Setup --
  [fs.read] read_file
  => ok
  [fs.read] read_file
  => ok
{
  "baseline_config": "Baseline recipe: base model HuggingFaceTB/SmolLM2-360M with QLoRA (rank=8, alpha=16, dropout=0.0, targets q/k/v/o projections), optimizer adamw_8bit, learning rate 2e-4, cosine scheduler, 10 warmup steps, 50 max steps, batch size 4 per device, gradient accumulation 2, max sequence length 512, seed 42. Data budget: 500 train examples (from 4000 available), 2000 eval examples on validation_matched. MNLI labels mapped to single-letter A/B/C with minimal prompt (premise/hypothesis/answer). Tunable parameters: LORA_R, LORA_ALPHA, LORA_DROPOUT, LORA_TARGET_MODULES, USE_QLORA, LEARNING_RATE, LR_SCHEDULER, WARMUP_STEPS, MAX_STEPS, PER_DEVICE_BATCH_SIZE, GRAD_ACCUM_STEPS, MAX_SEQ_LENGTH, OPTIM

**Above:** pre-rendered 25-trial log. Open `runs/current/trial_history.jsonl` after a live run for the same audit trail. Figures/models require no re-run to inspect — the log itself is the evidence.